# 设置agent名称
创建Agent时，LangChain允许用户指定其名称

In [ ]:
from langchain.agents import create_agent
from dotenv import load_dotenv
import os
from langchain_qwq import ChatQwen
from rich import print as rprint
from langchain.tools import tool

load_dotenv(override=True)
model = ChatQwen(
    model="deepseek-v4-flash",
    api_base=os.getenv("DASHSCOPE_API_BASE"),  # 国内 Key 必须用国内地址
)

@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """
    天气查询工具

    Args:
        city: 城市名称
    """
    return f"{city}的天气为晴朗，25°C。"

In [2]:
agent = create_agent(
    model=model,
    name = "myAgent"
)
response = agent.invoke({"messages": ["你好"]})
# rprint(response)
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================
Name: myAgent

你好！很高兴见到你，👋

有什么我可以帮你的吗？无论是想聊聊天、解答问题、还是需要一些信息或建议，我都在这里。尽管跟我说吧！😊


## 使用场景
name 在 `Multi-Agent` 场景中最常被提及，用于区分不同的 Agent。但它的作用并不局限于`多Agent编排`。在实际工程中，出现如下场景，通常都建议为 Agent 设置一个清晰且稳定的 name 。
1. 流式输出归因：在启用流式输出时，name 可用于 标识当前输出内容来自哪个Agent。
2. 消息身份标记：设置 name 后，Agent 产生的 `AIMessage` 会携带对应的name信息 。
3. 调试与trace可读性：在调试、日志分析和链路追踪过程中，name 可以作为 Agent 的稳定标识，帮助开发者 快速判断当前执行的是哪个 Agent 。
4. 组件化封装：有助于在模块注册、运行监控、日志归档和能力复用时保持一致的身份标识。
5. 前端展示以及运行状态可观测性

## 系统提示词
使用 create_agent 创建 Agent 时，需传入 模型 和 工具 、可选地传入 系统提示词 。提示词为Agent提供了任务背景、行为准则和操作指南。
系统指令，即SystemMessage，通过 system_prompt 设置，定义 Agent 行为。这个参数可以是 str 或者 SystemMessage类型 。
提示词设置有两种方式： `静态设置` 和 `动态设置` 。动态设置需要借助中间件。

In [ ]:
agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="""你是天气助手。
    工作流程：
    1. 理解用户的城市查询
    2. 使用 get_weather 工具获取数据
    3. 简洁清晰地回答
    输出格式：
        - 天气状况
        - 温度
        - 注意事项（如有）
    """
)


## 结构化输出

它允许Agent以特定、可预测的格式返回数据，而不是传统的自然语言响应。通过结构化输出，开发者可以直接获得 Pydantic模型 、 JSON对象 或 数据类 等结构化数据，这些数据能够被应用程序直接使用，无需复杂的解析过程。

### 结构化输出的4种策略
需通过 `response_format`参数 设置期望的输出模式（Schema）。

当模型生成结构化数据时，系统会自动捕获、验证并将结果存储在Agent状态的structured_response键中。

```python
def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],
        ProviderStrategy[StructuredResponseT],
        type[StructuredResponseT],
        None
    ]
)
```

### ProviderStrategy
使用模型提供商的 原生结构化输出功能 实现结构化输出。
- 这里所说的“原生结构化输出”指的是大语言模型（LLM）提供商通过其API直接提供的、在模型响应阶段就强制保证 `输出格式符合预定规范` 的能力，这种能力能够在模型生成内容的源头确保结构化准确性。

- 适用于支持原生结构化输出的模型，比如OpenAI、Anthropic Claude或xAI Grok等。

In [7]:

from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy
from langchain.messages import HumanMessage
from rich import print as rprint
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")

# 3.agent初始化
agent = create_agent(
    model=model,
    response_format=ProviderStrategy(ContactInfo)
)

# 4.调用
response = agent.invoke({
    "messages": [
        HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
    ]
})
rprint(response)
for msg in response["messages"]:
    msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='a9491ec5-b90c-49ba-ad96-a87106103f6b'
        ),
        AIMessage(
            content='{\n  "name": "小明",\n  "email": "shkstart@atguigu.com",\n  "phone": "12345678912"\n}',
            additional_kwargs={
                'parsed': None,
                'refusal': None,
                'reasoning_content': '我们根据schema，需要抽取name, email, 
phone。文本中：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912。所以name: 小明，email: 
shkstart@atguigu.com，phone: 12345678912。输出JSON数组，因为schema是列表形式。'
            },
            response_metadata={
                'token_usage': {
                    'completion_tokens': 108,
                    'prompt_tokens': 151,
                    'total_tokens': 259,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': None,
                        'reasoning_tokens': 73,
                        'rejected_prediction_tokens': None
                    },
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'dashscope',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': None,
                'id': 'chatcmpl-27b357bc-f0bf-9d16-b130-d202d87c9a4b',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--019f1cc8-d315-77e2-a3f3-65cb946d28f1-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 151,
                'output_tokens': 108,
                'total_tokens': 259,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {'reasoning': 73}
            }
        )
    ],
    'structured_response': ContactInfo(name='小明', email='shkstart@atguigu.com', phone='12345678912')
}

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================

{
  "name": "小明",
  "email": "shkstart@atguigu.com",
  "phone": "12345678912"
}


### ToolStrategy(推荐使用)
对于不支持原生结构化输出的模型，LangChain采用`ToolStrategy`工具调用的方式实现结构化输出。此策略兼容绝大多数 `支持工具调用` 的现代模型。

其核心原理：
1. 动态创建一个`虚拟工具`
2. 该工具的输入参数对应着期望的数据结构。
3. 当模型需要生成最终答案时，系统会引导模型 "调用"这个虚拟工具 ，从而间接产生符合要求的结构化数据。

In [9]:
from langchain.agents.structured_output import ToolStrategy
from langchain.tools import tool

# 2.Pydantic结构化方式定义
class ContactInfo(BaseModel):
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="手机号")

# 3.工具的定义（根据需要定义）
@tool
def search_tool(query: str) -> str:
    """这是一个搜索引擎。当大模型发现给定的上下文里缺少必要的联系人信息，
    需要去互联网上查询时，才会调用这个工具。
    """
    return f"搜索结果: 未找到关于 '{query}' 的更多额外信息。"

# 4.agent初始化
agent = create_agent(
    model=model,
    tools=[search_tool],
    response_format=ToolStrategy(ContactInfo)
)
result = agent.invoke({
    "messages": [{"role": "user", "content": "联系人信息: John Doe,john@atguigu.com, (010) 56253825"}]
})
# print(result)
rprint(result["structured_response"])

ContactInfo(name='John Doe', email='john@atguigu.com', phone='(010) 56253825')

###  type / AutoStrategy
官方文档不存在，源码存在。
当我们直接传入一个定义类型时，LangChain会自动包装为AutoStrategy，触发 自动选择策略 ：

    如果模型支持原生结构化输出（如OpenAI、Anthropic Claude或xAI Grok），则优先使用 ProviderStrategy；否则使用ToolStrategy。

In [ ]:
from langchain.agents.structured_output import AutoStrategy
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """联系人信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="邮箱")
    phone: str = Field(description="电话")

agent = create_agent(
    model=model,
    response_format=AutoStrategy(ContactInfo)
)

### None
默认配置，表示不以结构化输出，以 自然语言 响应用户问题